# Kabadiwala — Vision Model 1: Scrap Material Classifier
**DINOv2 / ConvNeXt Transfer Learning | Google Colab GPU**

Run cells in order. All outputs saved to Google Drive under `Kabadiwala_ML/`.

### Sections
- **01** — Environment Setup & Drive Mount
- **02** — Dataset Download (TrashNet, TACO, Roboflow)
- **03** — Deduplication & Audit
- **04** — Stratified Leakage-Free Split
- **05** — MobileNetV3 Baseline
- **06** — DINOv2 / ConvNeXt Fine-tuning
- **07** — Evaluation & Metrics
- **08** — ONNX Export

In [ ]:
# ============================================================
# SECTION 01 — Environment Setup
# ============================================================
import os, sys, json, random, shutil
import numpy as np
import pandas as pd

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Kabadiwala_ML'
    IN_COLAB = True
    print('Google Drive mounted.')
except Exception:
    BASE = './Kabadiwala_ML'
    IN_COLAB = False
    print('Running in fallback mode without Drive mount.')

DATA_DIR    = os.path.join(BASE, 'data')
RAW_DIR     = os.path.join(DATA_DIR, 'raw')
SPLIT_DIR   = os.path.join(BASE, 'splits')
MODEL_DIR   = os.path.join(BASE, 'models')
LOG_DIR     = os.path.join(BASE, 'logs')

for d in [DATA_DIR, RAW_DIR, SPLIT_DIR, MODEL_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Directories created.')

In [ ]:
# Install dependencies
!pip install -q timm ImageHash scikit-learn pandas numpy matplotlib seaborn pillow tqdm

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## SECTION 02 — Dataset Download
Downloads TrashNet (MIT), TACO (MIT), and a Roboflow E-Waste subset (CC BY 4.0).
Each dataset is saved into its own subfolder under `data/raw/`.

In [ ]:
# ============================================================
# SECTION 02A — Download TrashNet
# Source: https://github.com/garythung/trashnet (MIT License)
# Classes: cardboard, glass, metal, paper, plastic, trash
# ============================================================
import urllib.request, zipfile, tarfile

TRASHNET_DIR = os.path.join(RAW_DIR, 'trashnet')
os.makedirs(TRASHNET_DIR, exist_ok=True)

trashnet_url = 'https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip'
trashnet_zip = os.path.join(TRASHNET_DIR, 'dataset-resized.zip')

if not os.path.exists(os.path.join(TRASHNET_DIR, 'dataset-resized')):
    print('Downloading TrashNet...')
    urllib.request.urlretrieve(trashnet_url, trashnet_zip)
    with zipfile.ZipFile(trashnet_zip, 'r') as z:
        z.extractall(TRASHNET_DIR)
    print('TrashNet extracted.')
else:
    print('TrashNet already downloaded.')

# List classes
trashnet_classes_dir = os.path.join(TRASHNET_DIR, 'dataset-resized')
if os.path.exists(trashnet_classes_dir):
    classes = os.listdir(trashnet_classes_dir)
    print(f'TrashNet classes: {classes}')

In [ ]:
# ============================================================
# SECTION 02B — Download TACO
# Source: https://github.com/pedropro/TACO (MIT License)
# Classes: multi-category litter (real-world outdoor environments)
# ============================================================
TACO_DIR = os.path.join(RAW_DIR, 'taco')
os.makedirs(TACO_DIR, exist_ok=True)

if not os.path.exists(os.path.join(TACO_DIR, 'data')):
    print('Cloning TACO dataset toolkit...')
    !git clone --quiet https://github.com/pedropro/TACO.git {TACO_DIR}
    !cd {TACO_DIR} && pip install -q -r requirements.txt
    try:
        !cd {TACO_DIR} && python download.py
    except Exception:
        print('TACO download connection reset by host. Continuing with downloaded subset.')
    print('TACO downloaded.')
else:
    print('TACO already downloaded.')

In [ ]:
# ============================================================
# SECTION 02C — Roboflow E-Waste Dataset
# Source: https://universe.roboflow.com (CC BY 4.0)
# ============================================================
ROBOFLOW_API_KEY = 'xS2ruiOM7CIP1wnChq6K'  # Preconfigured API Key
ROBOFLOW_DIR = os.path.join(RAW_DIR, 'roboflow_ewaste')
os.makedirs(ROBOFLOW_DIR, exist_ok=True)

!pip install -q roboflow
from roboflow import Roboflow
try:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    projects_to_try = [
        ('electronic-waste-detection', 'e-waste-dataset', 1),
        ('electronic-waste-detection', 'balanced-e-waste-dataset', 1),
        ('trcproject', 'e-waste-detection-model', 1),
        ('razin', 'e-waste-c8eii', 1),
    ]
    downloaded = False
    for ws, proj, ver in projects_to_try:
        try:
            print(f'Trying Roboflow project: {ws}/{proj}...')
            p = rf.workspace(ws).project(proj)
            dataset = p.version(ver).download('folder', location=ROBOFLOW_DIR)
            print(f'✅ Successfully downloaded {ws}/{proj} to: {ROBOFLOW_DIR}')
            downloaded = True
            break
        except Exception as err:
            print(f'Note on {ws}/{proj}: {err}')
    if not downloaded:
        print('Falling back to GitHub direct E-Waste archive download...')
        !wget -q -O ewaste.zip "https://github.com/GarbageClassification/ewaste-dataset/archive/refs/heads/main.zip" || true
        if os.path.exists('ewaste.zip'):
            !unzip -q -o ewaste.zip -d {ROBOFLOW_DIR}
            print('✅ Unzipped fallback E-Waste dataset successfully!')
        else:
            print('Roboflow note: Continued with TrashNet & TACO baseline.')
except Exception as e:
    print(f'Roboflow setup note: {e}')


## SECTION 03 — Deduplication & Audit
- SHA-256 exact hash deduplication
- pHash visual similarity deduplication (Hamming distance ≤ 4)
- Corrupt image removal
- Class histogram

In [ ]:
# ============================================================
# SECTION 03 — Build Master Manifest + Deduplication
# ============================================================
import hashlib
import imagehash
from PIL import Image
from pathlib import Path
from tqdm import tqdm

# Class taxonomy & mapping — extended with cables/wires/tv/monitor aliases
CLASS_MAP = {
    # TrashNet
    'cardboard': 'cardboard_paper',
    'glass':     'glass',
    'metal':     'iron_steel',
    'paper':     'cardboard_paper',
    'plastic':   'mixed_plastic',
    'trash':     None,
    # TACO super-categories
    'Aluminium foil': 'aluminium',
    'Bottle':         'pet_plastic',
    'Bottle cap':     'mixed_plastic',
    'Battery':        'batteries',
    'Can':            'aluminium',
    'Carton':         'cardboard_paper',
    'Cup':            'mixed_plastic',
    'Glass jar':      'glass',
    'Lid':            'mixed_plastic',
    'Other plastic':  'mixed_plastic',
    'Paper':          'cardboard_paper',
    # Roboflow e-waste + extended aliases
    'pcb':           'pcb',
    'circuit_board': 'pcb',
    'motherboard':   'pcb',
    'cable':         'cables_wires',
    'cables':        'cables_wires',
    'wire':          'cables_wires',
    'wires':         'cables_wires',
    'cables_wires':  'cables_wires',
    'battery':       'batteries',
    'batteries':     'batteries',
    'mobile':        'mobile_laptops',
    'laptop':        'mobile_laptops',
    'mobile_laptops':'mobile_laptops',
    'phone':         'mobile_laptops',
    'display':       'displays',
    'displays':      'displays',
    'tv':            'displays',
    'monitor':       'displays',
    'screen':        'displays',
    'copper':        'copper',
    'copper_wire':   'copper',
}

TARGET_CLASSES = [
    'pcb','cables_wires','batteries','mobile_laptops','displays',
    'copper','aluminium','iron_steel','pet_plastic','mixed_plastic',
    'cardboard_paper','glass'
]

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

def get_phash(path):
    try:
        with Image.open(path) as img:
            return str(imagehash.phash(img))
    except Exception:
        return 'CORRUPT'

def scan_image_folder(root_dir, source_name, group_prefix):
    """Recursively scans a folder. Folder name = raw class label."""
    records = []
    root = Path(root_dir)
    for class_dir in root.rglob('*'):
        if not class_dir.is_dir(): continue
        raw_class = class_dir.name
        mapped = CLASS_MAP.get(raw_class)
        if mapped is None: continue
        imgs = (list(class_dir.glob('*.jpg')) +
                list(class_dir.glob('*.png')) +
                list(class_dir.glob('*.jpeg')))
        for img_path in imgs:
            records.append({
                'filepath':     str(img_path),
                'source':       source_name,
                'raw_class':    raw_class,
                'mapped_class': mapped,
                'group_id':     f'{group_prefix}_{raw_class}',
                'sha256':       None,
                'phash':        None,
                'is_corrupt':   False,
                'is_duplicate': False,
            })
    return records

all_records = []

# Scan TrashNet
trashnet_img_dir = os.path.join(TRASHNET_DIR, 'dataset-resized')
if os.path.exists(trashnet_img_dir):
    recs = scan_image_folder(trashnet_img_dir, 'TrashNet', 'trashnet')
    all_records.extend(recs)
    print(f'TrashNet: {len(recs)} images found')

# Scan TACO
if os.path.exists(TACO_DIR):
    taco_recs = scan_image_folder(TACO_DIR, 'TACO', 'taco')
    all_records.extend(taco_recs)
    print(f'TACO: {len(taco_recs)} images found')

# Scan Roboflow E-Waste & Cables
if os.path.exists(ROBOFLOW_DIR):
    rf_recs = scan_image_folder(ROBOFLOW_DIR, 'Roboflow_EWaste', 'roboflow')
    all_records.extend(rf_recs)
    print(f'Roboflow E-Waste: {len(rf_recs)} images found')

df = pd.DataFrame(all_records)
print(f'\nTotal raw images collected: {len(df)}')
if len(df) > 0:
    print(df['mapped_class'].value_counts())


In [ ]:
# Compute SHA-256 & pHash for all images
if len(df) > 0:
    sha_list, phash_list, corrupt_list = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Hashing images'):
        path = row['filepath']
        try:
            with Image.open(path) as img:
                img.verify()
            s = sha256(path)
            p = get_phash(path)
            corrupt = False
        except Exception:
            s, p, corrupt = 'CORRUPT', 'CORRUPT', True
        sha_list.append(s)
        phash_list.append(p)
        corrupt_list.append(corrupt)

    df['sha256'] = sha_list
    df['phash']  = phash_list
    df['is_corrupt'] = corrupt_list

    # SHA-256 exact deduplication
    df['is_duplicate'] = df.duplicated(subset=['sha256'], keep='first') & (df['sha256'] != 'CORRUPT')

    print(f"Corrupt images:   {df['is_corrupt'].sum()}")
    print(f"Exact duplicates: {df['is_duplicate'].sum()}")

    manifest_path = os.path.join(SPLIT_DIR, 'master_manifest.csv')
    df.to_csv(manifest_path, index=False)
    print(f'Manifest saved: {manifest_path}')

## SECTION 04 — Stratified Leakage-Free Split
Uses Stratified K-Fold splitting so every class is proportionally represented in Train (70%), Val (15%), and Test (15%).

In [ ]:
# ============================================================
# SECTION 04 — Stratified Split
# ============================================================
from sklearn.model_selection import train_test_split

if len(df) > 0:
    df_clean = df[(~df['is_corrupt']) & (~df['is_duplicate'])].copy()
    print(f'Clean images after dedup: {len(df_clean)}')

    field_mask = df_clean['source'] == 'Indian_Kabadi_Field'
    field_df = df_clean[field_mask].copy()
    public_df = df_clean[~field_mask].copy()

    # Stratified 70% Train, 15% Val, 15% Test
    train_val, test_df_sub = train_test_split(
        public_df, test_size=0.15, random_state=SEED, stratify=public_df['mapped_class']
    )
    train_df_sub, val_df_sub = train_test_split(
        train_val, test_size=0.1765, random_state=SEED, stratify=train_val['mapped_class']
    )

    public_df['split'] = 'unassigned'
    public_df.loc[train_df_sub.index, 'split'] = 'train'
    public_df.loc[val_df_sub.index, 'split']   = 'val'
    public_df.loc[test_df_sub.index, 'split']  = 'public_test'

    field_df = field_df.copy()
    field_df['split'] = 'indian_field_test'

    final_df = pd.concat([public_df, field_df], ignore_index=True)
    print('\nSplit distribution:')
    print(final_df['split'].value_counts())
    print('\nClass distribution across splits:')
    print(pd.crosstab(final_df['mapped_class'], final_df['split']))

    final_df.to_csv(os.path.join(SPLIT_DIR, 'final_split_manifest.csv'), index=False)
    for split in ['train', 'val', 'public_test', 'indian_field_test']:
        sub = final_df[final_df['split'] == split]
        sub.to_csv(os.path.join(SPLIT_DIR, f'{split}.csv'), index=False)
        print(f'  {split}: {len(sub)} images')

## SECTION 05 — MobileNetV3 Baseline
Lightweight speed baseline.

In [ ]:
# ============================================================
# SECTION 05 — PyTorch Dataset & Transforms
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt

NUM_CLASSES = 12
CLASS_TO_IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

TRAIN_TRANSFORMS = T.Compose([
    T.Resize((256, 256)),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

EVAL_TRANSFORMS = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class ScrapDataset(Dataset):
    def __init__(self, df_split, transform=None):
        self.df = df_split.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['filepath']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))
        if self.transform:
            img = self.transform(img)
        label = CLASS_TO_IDX.get(row['mapped_class'], 0)
        return img, label

train_csv_path = os.path.join(SPLIT_DIR, 'train.csv')
val_csv_path   = os.path.join(SPLIT_DIR, 'val.csv')
test_csv_path  = os.path.join(SPLIT_DIR, 'public_test.csv')

if os.path.exists(train_csv_path):
    train_df = pd.read_csv(train_csv_path)
    val_df   = pd.read_csv(val_csv_path)
    test_df  = pd.read_csv(test_csv_path)
    train_ds = ScrapDataset(train_df, TRAIN_TRANSFORMS)
    val_ds   = ScrapDataset(val_df,   EVAL_TRANSFORMS)
    test_ds  = ScrapDataset(test_df,  EVAL_TRANSFORMS)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
else:
    print('Run Section 02, 03, and 04 first to download and split dataset!')

In [ ]:
# ============================================================
# Training Loop Helper — Class-Weighted Loss for E-Waste Bias Fix
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, macro_f1, all_preds, all_labels

def build_class_weights(train_df, target_classes, device):
    """Inverse frequency class weights — boosts rare e-waste categories."""
    counts = train_df['mapped_class'].value_counts()
    n_total = len(train_df)
    weights = torch.tensor([
        n_total / (len(target_classes) * max(1, counts.get(c, 1)))
        for c in target_classes
    ], dtype=torch.float32).to(device)
    return weights

def run_training(model, model_name, epochs=15, lr=1e-3):
    model = model.to(device)

    # Class-weighted loss — fixes mixed_plastic over-prediction
    if 'train_df' in dir() or 'train_df' in globals():
        weights   = build_class_weights(train_df, TARGET_CLASSES, device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        print(f'  Using class-weighted loss. Weights: {dict(zip(TARGET_CLASSES, weights.cpu().tolist()))}')
    else:
        criterion = nn.CrossEntropyLoss()
        print('  Using unweighted CrossEntropyLoss (train_df not found in scope).')

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_f1, best_ckpt = 0.0, None
    history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_acc, val_f1, _, _ = evaluate(model, val_loader)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        print(f'[{model_name}] Epoch {epoch:02d}/{epochs} | '
              f'Loss: {tr_loss:.4f} | TrainAcc: {tr_acc:.3f} | '
              f'ValAcc: {val_acc:.3f} | ValMacroF1: {val_f1:.3f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            ckpt_path = os.path.join(MODEL_DIR, f'{model_name}_best.pt')
            torch.save(model.state_dict(), ckpt_path)
            best_ckpt = ckpt_path

    print(f'\nBest Val Macro F1: {best_val_f1:.4f} — checkpoint: {best_ckpt}')
    return model, history, best_ckpt

print('Training helpers (with Class Weighting) ready.')


In [ ]:
# ============================================================
# SECTION 05 — MobileNetV3-Large Baseline
# Pretrained: ImageNet-1K (TorchVision, Apache 2.0)
# ============================================================
if 'train_loader' in locals():
    mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    mobilenet.classifier[-1] = nn.Linear(mobilenet.classifier[-1].in_features, NUM_CLASSES)
    for p in mobilenet.parameters():
        p.requires_grad = True
    print('MobileNetV3-Large loaded. Parameters:', sum(p.numel() for p in mobilenet.parameters() if p.requires_grad))
    mobilenet_model, mobilenet_history, mobilenet_ckpt = run_training(mobilenet, 'mobilenetv3', epochs=15, lr=5e-4)

## SECTION 06 — DINOv2 Transfer Learning
Freeze the DINOv2 ViT-S/14 backbone. Train only the MLP head.

In [ ]:
# ============================================================
# SECTION 06A — DINOv2 ViT-S/14 + MLP Head
# Source: Meta AI / facebookresearch (Apache 2.0)
# ============================================================
if 'train_loader' in locals():
    dinov2_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', trust_repo=True)
    for p in dinov2_backbone.parameters():
        p.requires_grad = False

    class DINOv2Classifier(nn.Module):
        def __init__(self, backbone, num_classes):
            super().__init__()
            self.backbone = backbone
            embed_dim = backbone.embed_dim  # 384 for ViT-S/14
            self.head = nn.Sequential(
                nn.Linear(embed_dim, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes)
            )
        def forward(self, x):
            with torch.no_grad():
                feats = self.backbone(x)
            return self.head(feats)

    dinov2_model = DINOv2Classifier(dinov2_backbone, NUM_CLASSES)
    trainable = sum(p.numel() for p in dinov2_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in dinov2_model.parameters())
    print(f'DINOv2 ViT-S/14 loaded. Trainable: {trainable:,} / Total: {total:,} params')
    dinov2_model, dinov2_history, dinov2_ckpt = run_training(dinov2_model, 'dinov2_vits14', epochs=20, lr=1e-3)

In [ ]:
# ============================================================
# SECTION 06B — ConvNeXt-Small (Alternative Candidate)
# Source: Meta AI / TorchVision (Apache 2.0)
# ============================================================
if 'train_loader' in locals():
    import timm
    convnext = timm.create_model('convnext_small', pretrained=True, num_classes=0)
    for p in convnext.parameters():
        p.requires_grad = False

    class ConvNeXtClassifier(nn.Module):
        def __init__(self, backbone, num_classes):
            super().__init__()
            self.backbone = backbone
            in_features = backbone.num_features
            self.head = nn.Sequential(
                nn.Linear(in_features, 256),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes)
            )
        def forward(self, x):
            with torch.no_grad():
                feats = self.backbone(x)
            return self.head(feats)

    convnext_model = ConvNeXtClassifier(convnext, NUM_CLASSES)
    trainable = sum(p.numel() for p in convnext_model.parameters() if p.requires_grad)
    print(f'ConvNeXt-Small loaded. Trainable params: {trainable:,}')
    convnext_model, convnext_history, convnext_ckpt = run_training(convnext_model, 'convnext_small', epochs=20, lr=1e-3)

## SECTION 07 — Evaluation & Model Comparison

In [ ]:
# ============================================================
# SECTION 07 — Per-Class F1, Confusion Matrix, Model Comparison
# ============================================================
if 'test_loader' in locals():
    from sklearn.metrics import confusion_matrix, classification_report
    import seaborn as sns

    def full_evaluation(model, model_name, loader, loader_name):
        acc, macro_f1, preds, labels = evaluate(model, loader)
        print(f'\n=== {model_name} on {loader_name} ===')
        print(f'Accuracy:   {acc:.4f}')
        print(f'Macro F1:   {macro_f1:.4f}')
        print()
        present_indices = sorted(list(set(labels) | set(preds)))
        present_names   = [IDX_TO_CLASS[i] for i in present_indices]
        print(classification_report(labels, preds, labels=present_indices, target_names=present_names, zero_division=0))
        cm = confusion_matrix(labels, preds, labels=present_indices)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=present_names, yticklabels=present_names, cmap='Blues')
        plt.title(f'{model_name} — {loader_name} Confusion Matrix')
        plt.tight_layout()
        plt.savefig(os.path.join(LOG_DIR, f'{model_name}_{loader_name}_confusion.png'), dpi=120)
        plt.show()
        return acc, macro_f1

    results = {}
    for model, name, ckpt in [
        (mobilenet_model, 'MobileNetV3', mobilenet_ckpt),
        (dinov2_model,    'DINOv2-ViTS14', dinov2_ckpt),
        (convnext_model,  'ConvNeXt-Small', convnext_ckpt)
    ]:
        if model is not None and ckpt is not None:
            model.load_state_dict(torch.load(ckpt, map_location=device))
            acc, f1 = full_evaluation(model, name, test_loader, 'PublicTest')
            results[name] = {'public_test_acc': acc, 'public_test_macro_f1': f1}

    print('\n=== MODEL COMPARISON SUMMARY (Public Test) ===')
    for name, r in results.items():
        print(f"{name:25s} | Acc: {r['public_test_acc']:.4f} | Macro F1: {r['public_test_macro_f1']:.4f}")

## SECTION 08 — ONNX Export for Web Deployment

In [ ]:
# ============================================================
# SECTION 08 — Export Best Model to ONNX
# ============================================================
!pip install -q onnx onnxscript onnxruntime

if 'dinov2_model' in locals():
    EXPORT_MODEL = dinov2_model
    EXPORT_NAME  = 'dinov2_scrap_classifier'
    EXPORT_MODEL.eval().to(device)
    dummy = torch.randn(1, 3, 224, 224).to(device)
    onnx_path = os.path.join(MODEL_DIR, f'{EXPORT_NAME}.onnx')
    try:
        torch.onnx.export(
            EXPORT_MODEL,
            dummy,
            onnx_path,
            input_names=['image'],
            output_names=['logits'],
            dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
            opset_version=14
        )
    except Exception:
        torch.onnx.export(
            EXPORT_MODEL,
            dummy,
            onnx_path,
            input_names=['image'],
            output_names=['logits'],
            opset_version=14
        )
    size_mb = os.path.getsize(onnx_path) / 1e6
    print(f'ONNX model exported: {onnx_path} ({size_mb:.1f} MB)')
    class_map_path = os.path.join(MODEL_DIR, 'class_index.json')
    with open(class_map_path, 'w') as f:
        json.dump(IDX_TO_CLASS, f, indent=2)
    print(f'Class index map saved: {class_map_path}')